In [1]:
!pip -q install -U "transformers>=4.45.0" accelerate bitsandbytes gradio huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.


In [4]:
import os, getpass
from huggingface_hub import login

hf_token = getpass.getpass("Masukkan Hugging Face User Access Token (tidak akan ditampilkan): ")
login(token=hf_token, add_to_git_credential=False)
os.environ["HF_TOKEN"] = hf_token

admin_password = getpass.getpass("Buat password admin baru (minimal 12 karakter): ")
if len(admin_password) < 12:
    raise ValueError("Password admin minimal 12 karakter.")
os.environ["JASTIP_ADMIN_PASSWORD"] = admin_password
print("Login Hugging Face berhasil dan password admin disetel untuk sesi ini.")

Masukkan Hugging Face User Access Token (tidak akan ditampilkan): ··········


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Buat password admin baru (minimal 12 karakter): ··········
Login Hugging Face berhasil dan password admin disetel untuk sesi ini.


In [17]:
%%writefile app.py
import os, sqlite3, hashlib, hmac, secrets, datetime, traceback
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

APP_NAME = "Jastip Hallo Dek"
DB_PATH = "jastip_hallo_dek.db"
MODEL_ID = "google/gemma-2-2b-it"
STATUSES = ["Menunggu konfirmasi", "Menunggu pembayaran", "Diproses",
            "Memuat pesanan", "Dalam pengantaran", "Selesai", "Dibatalkan"]
PRODUCTS = {
    "Birthday Cake": {"price": 100000, "description": "Kue ulang tahun untuk momen spesial", "emoji": "🎂"},
    "Love Bouquet": {"price": 75000, "description": "Buket bunga manis untuk orang tersayang", "emoji": "💐"},
    "Custom Love Pillow": {"price": 120000, "description": "Bantal custom dengan pesan personal", "emoji": "🧸"},
}
DELIVERY_FEE = 20000
ADMIN_PASSWORD = os.environ.get("JASTIP_ADMIN_PASSWORD", "GantiPasswordAdmin123!")

def db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    with db() as c:
        c.execute("""CREATE TABLE IF NOT EXISTS users(
            id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT NOT NULL,
            email TEXT UNIQUE NOT NULL, password_hash TEXT NOT NULL,
            salt TEXT NOT NULL, role TEXT NOT NULL DEFAULT 'customer',
            created_at TEXT NOT NULL)""")
        c.execute("""CREATE TABLE IF NOT EXISTS orders(
            id INTEGER PRIMARY KEY AUTOINCREMENT, customer_email TEXT NOT NULL,
            customer_name TEXT NOT NULL, recipient TEXT NOT NULL,
            recipient_phone TEXT NOT NULL, product TEXT NOT NULL,
            quantity INTEGER NOT NULL, address TEXT NOT NULL,
            delivery_date TEXT NOT NULL, message TEXT, total INTEGER NOT NULL,
            status TEXT NOT NULL, created_at TEXT NOT NULL)""")

def money(n):
    return "Rp" + f"{int(n):,}".replace(",", ".")

def pw_hash(password, salt):
    return hashlib.pbkdf2_hmac("sha256", password.encode(), bytes.fromhex(salt), 180000).hex()

def register(name, email, password):
    name, email = name.strip(), email.strip().lower()
    if not name or not email or len(password or "") < 8:
        return None, "Isi nama dan email. Password minimal 8 karakter."
    salt = secrets.token_hex(16)
    try:
        with db() as c:
            c.execute("INSERT INTO users(name,email,password_hash,salt,role,created_at) VALUES(?,?,?,?,?,?)",
                      (name,email,pw_hash(password,salt),salt,"customer",datetime.datetime.now().isoformat(timespec="seconds")))
        return {"name":name,"email":email,"role":"customer"}, "Pendaftaran berhasil. Kamu sudah login."
    except sqlite3.IntegrityError:
        return None, "Email sudah terdaftar."

def login(email, password):
    with db() as c:
        row = c.execute("SELECT * FROM users WHERE email=?", (email.strip().lower(),)).fetchone()
    if not row or not hmac.compare_digest(row["password_hash"], pw_hash(password, row["salt"])):
        return None, "Email atau password salah."
    return {"name":row["name"],"email":row["email"],"role":row["role"]}, f"Berhasil masuk sebagai {row['name']}."

def create_order(user, recipient, phone, product, quantity, address, delivery_date, message):
    if not user:
        return "Silakan daftar atau login terlebih dahulu."
    if not recipient or not phone or not address or not delivery_date:
        return "Lengkapi nama penerima, nomor telepon, alamat, dan tanggal pengiriman."
    quantity = int(quantity)
    if quantity < 1 or quantity > 10:
        return "Jumlah pesanan harus 1–10."
    total = PRODUCTS[product]["price"] * quantity + DELIVERY_FEE
    with db() as c:
        cur = c.execute("""INSERT INTO orders(customer_email,customer_name,recipient,recipient_phone,
            product,quantity,address,delivery_date,message,total,status,created_at)
            VALUES(?,?,?,?,?,?,?,?,?,?,?,?)""",
            (user["email"],user["name"],recipient,phone,product,quantity,address,
             str(delivery_date),message or "",total,STATUSES[0],
             datetime.datetime.now().isoformat(timespec="seconds")))
        oid = cur.lastrowid
    return f"Pesanan berhasil dibuat! Nomor pesanan: JHD-{oid:05d}. Total: {money(total)}."

def customer_orders(user):
    if not user:
        return []
    with db() as c:
        rows = c.execute("""SELECT id,product,quantity,recipient,delivery_date,total,status,created_at
                            FROM orders WHERE customer_email=? ORDER BY id DESC""",(user["email"],)).fetchall()
    return [[f"JHD-{r['id']:05d}",r["product"],r["quantity"],r["recipient"],r["delivery_date"],
             money(r["total"]),r["status"],r["created_at"]] for r in rows]

def admin_login(password):
    if not hmac.compare_digest(password or "", ADMIN_PASSWORD):
        return False, "Password admin salah."
    return True, "Akses admin berhasil. Jangan bagikan password admin."

def admin_orders():
    with db() as c:
        rows = c.execute("SELECT * FROM orders ORDER BY id DESC").fetchall()
    return [[f"JHD-{r['id']:05d}",r["customer_name"],r["customer_email"],r["product"],
             r["quantity"],r["recipient"],r["delivery_date"],money(r["total"]),r["status"],r["id"]] for r in rows]

def update_order_status(order_id, status, admin_ok):
    if not admin_ok:
        return "Login admin terlebih dahulu."
    try:
        oid = int(str(order_id).replace("JHD-",""))
    except Exception:
        return "Nomor pesanan tidak valid."
    if status not in STATUSES:
        return "Status tidak valid."
    with db() as c:
        cur = c.execute("UPDATE orders SET status=? WHERE id=?", (status,oid))
    return "Status pesanan diperbarui." if cur.rowcount else "Pesanan tidak ditemukan."

tokenizer = None
model = None
def load_gemma():
    global tokenizer, model
    if tokenizer is not None and model is not None:
        return "Gemma sudah siap."
    token = os.environ.get("HF_TOKEN")
    if not token:
        return "HF_TOKEN belum tersedia. Jalankan sel login Hugging Face di notebook."
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
    if not torch.cuda.is_available():
        raise RuntimeError("GPU tidak terdeteksi. Di Google Colab pilih Runtime > Change runtime type > T4 GPU, lalu jalankan ulang.")
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=torch.float16,
                               bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, token=token, device_map="auto",
        quantization_config=quant, torch_dtype=torch.float16)
    model.eval()
    return "Gemma 2 2B Instruct berhasil dimuat (4-bit)."

SYSTEM_PROMPT = """Kamu adalah asisten layanan pelanggan Jastip Hallo Dek. Jawab dalam bahasa Indonesia yang ramah, jelas, singkat, dan sopan.
Bantu pelanggan memahami katalog: Birthday Cake Rp100.000, Love Bouquet Rp75.000, Custom Love Pillow Rp120.000. Ongkir contoh Rp20.000.
Jelaskan bahwa harga dan ongkir adalah contoh demo yang perlu dikonfirmasi admin. Jangan mengarang status pesanan, ketersediaan, estimasi pengiriman, atau pembayaran.
Untuk membuat pesanan, arahkan pelanggan ke tab Buat Pesanan. Untuk status pesanan, arahkan ke tab Pesanan Saya."""

def chatbot(message, history):
    if not message or not message.strip():
        return "Tulis pertanyaan terlebih dahulu."
    if model is None or tokenizer is None:
        return "Model AI belum siap. Pastikan token Hugging Face sudah dimasukkan, akses Gemma sudah disetujui, dan sel pemuatan model berhasil dijalankan."
    msgs = []
    for item in (history or [])[-8:]:
        if isinstance(item, dict) and item.get("role") in ("user","assistant"):
            msgs.append({"role":item["role"],"content":item.get("content","")})
        elif isinstance(item, (tuple,list)) and len(item)==2:
            msgs.extend([{"role":"user","content":str(item[0])},{"role":"assistant","content":str(item[1])}])
    msgs.append({"role":"user","content":message})
    if msgs and msgs[0]["role"] == "user":
        msgs[0]["content"] = SYSTEM_PROMPT + "\n\n" + msgs[0]["content"]
    else:
        msgs.insert(0, {"role":"user","content":SYSTEM_PROMPT})
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=220, do_sample=True,
                                temperature=0.7, top_p=0.9,
                                pad_token_id=tokenizer.eos_token_id)
    answer = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    return answer or "Maaf, saya belum bisa menjawab. Coba tulis pertanyaan dengan cara lain."

init_db()
try:
    load_gemma()
except Exception as e:
    print("Gemma belum dimuat otomatis:", e)

with gr.Blocks(theme=gr.themes.Soft(primary_hue="rose", secondary_hue="emerald"),
               title=APP_NAME, css="""
               .gradio-container {max-width: 1100px !important}
               .hero {text-align:center;padding:18px;border-radius:18px;background:linear-gradient(120deg,#f8e8eb,#e8efe8)}
               """) as demo:
    user_state = gr.State(None)
    admin_state = gr.State(False)

    gr.HTML("""
    <div class="website">
        <header class="navbar">
            <div class="brand">
                <span class="brand-icon">H</span>
                <div>
                    HALLO DEK
                    <small>GIFT STUDIO</small>
                </div>
            </div>
            <nav>
                <a href="#home">BERANDA</a>
                <a href="#collection">KOLEKSI</a>
                <a href="#order">CARA PESAN</a>
            </nav>
            <div class="nav-love">MADE WITH LOVE ♡</div>
        </header>

        <section class="hero" id="home">
            <div class="hero-content">
                <span class="eyebrow">A LITTLE THOUGHT, A LOT OF LOVE</span>
                <h1>Hadiah yang<br><i>berbicara</i><br>untuk hatimu.</h1>
                <p>Rayakan setiap momen istimewa dengan hadiah pilihan yang dirangkai penuh perhatian. Kami bantu sampaikan rasa sayangmu.</p>
                <a href="#collection" class="hero-button">JELAJAHI KOLEKSI <span>↗</span></a>
                <div class="hero-bottom">
                    <span>HALLO DEK — GIFT STUDIO</span>
                    <span>01 / 03</span>
                </div>
            </div>
            <div class="hero-image"></div>
        </section>

        <div class="running-text">
            ♡ THOUGHTFUL GIFTS &nbsp;&nbsp;&nbsp; ✿ MADE WITH LOVE &nbsp;&nbsp;&nbsp; ♡ DELIVERED WITH CARE
        </div>

        <section class="collection-intro" id="collection">
            <div>
                <span class="eyebrow">OUR LITTLE COLLECTION</span>
                <h2>Temukan cara manis<br>untuk <i>mengungkapkan rasa.</i></h2>
            </div>
            <p>Dari kejutan sederhana hingga hadiah yang tak terlupakan. Pilih sesuatu yang paling mewakili perasaanmu.</p>
        </section>
    </div>
    """)

    with gr.Tabs():
        with gr.Tab("✿ Koleksi"):
            gr.HTML("""
            <div class="catalog-heading">
                <span>01 — THE COLLECTION</span>
                <h2>Dipilih dengan hati.</h2>
                <p>Hadiah spesial untuk orang yang spesial.</p>
            </div>
            """)

            product_images = {
                "Birthday Cake": "https://images.unsplash.com/photo-1578985545062-69928b1d9587?w=700",
                "Love Bouquet": "https://images.unsplash.com/photo-1490750967868-88aa4486c946?w=700",
                "Custom Love Pillow": "https://images.unsplash.com/photo-1584100936595-c0654b55a2e2?w=700"
            }
            product_cards = ""
            for pname, info in PRODUCTS.items():
                product_cards += f"""
                <div class="product-card">
                    <div class="product-image">
                        <img src="{product_images[pname]}" alt="{pname}">
                        <span class="product-tag">MADE WITH LOVE</span>
                    </div>
                    <div class="product-info">
                        <h3>{pname}</h3>
                        <p>{info['description']}</p>
                        <div class="product-price">{money(info['price'])}</div>
                    </div>
                </div>
                """
            gr.HTML(f"""
            <div class="product-grid">
                {product_cards}
            </div>
            <div class="delivery-note">
                ✿ &nbsp; Pengiriman hadiah dengan penuh perhatian.
            </div>
            """)

        with gr.Tab("Akun"):
            gr.Markdown("## Daftar akun / Masuk")
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### Daftar")
                    reg_name = gr.Textbox(label="Nama lengkap")
                    reg_email = gr.Textbox(label="Email")
                    reg_pw = gr.Textbox(label="Password (min. 8 karakter)", type="password")
                    reg_btn = gr.Button("Buat akun", variant="primary")
                with gr.Column():
                    gr.Markdown("### Masuk")
                    log_email = gr.Textbox(label="Email")
                    log_pw = gr.Textbox(label="Password", type="password")
                    log_btn = gr.Button("Masuk")
            account_msg = gr.Markdown()
            current_user = gr.Markdown("Status: belum login.")
            reg_btn.click(register,[reg_name,reg_email,reg_pw],[user_state,account_msg]).then(
                lambda u: f"Login sebagai **{u['name']} ({u['role']})**" if u else "Status: belum login.", user_state,current_user)
            log_btn.click(login,[log_email,log_pw],[user_state,account_msg]).then(
                lambda u: f"Login sebagai **{u['name']} ({u['role']})**" if u else "Status: belum login.", user_state,current_user)

        with gr.Tab("Buat Pesanan"):
            gr.Markdown("## Formulir pemesanan")
            recipient = gr.Textbox(label="Nama penerima")
            phone = gr.Textbox(label="Nomor telepon penerima")
            product = gr.Dropdown(choices=list(PRODUCTS), value="Love Bouquet", label="Produk")
            quantity = gr.Slider(1,10,step=1,value=1,label="Jumlah")
            address = gr.Textbox(label="Alamat pengiriman", lines=2)
            date = gr.Textbox(label="Tanggal pengiriman (contoh: 2026-11-10)", placeholder="YYYY-MM-DD")
            message = gr.Textbox(label="Pesan pada hadiah (opsional)", lines=3)
            estimate = gr.Markdown(f"**Ongkir demo:** {money(DELIVERY_FEE)}")
            product.change(lambda p,q:f"**Estimasi total: {money(PRODUCTS[p]['price']*int(q)+DELIVERY_FEE)}**", [product,quantity], estimate)
            quantity.change(lambda p,q:f"**Estimasi total: {money(PRODUCTS[p]['price']*int(q)+DELIVERY_FEE)}**", [product,quantity], estimate)
            order_btn = gr.Button("Kirim Pesanan", variant="primary")
            order_msg = gr.Markdown()
            order_btn.click(create_order,[user_state,recipient,phone,product,quantity,address,date,message],order_msg)

        with gr.Tab("Pesanan Saya"):
            gr.Markdown("## Riwayat dan status pesanan")
            refresh_customer = gr.Button("Muat ulang pesanan")
            customer_table = gr.Dataframe(headers=["Nomor","Produk","Jumlah","Penerima","Tanggal","Total","Status","Dibuat"], interactive=False)
            refresh_customer.click(customer_orders,user_state,customer_table)
            gr.Markdown("Login terlebih dahulu, lalu klik Muat ulang pesanan.")

        with gr.Tab("Asisten AI"):
            gr.Markdown("## 💬 Hallo Dek AI")
            gr.Markdown("Tanyakan produk, proses pemesanan, atau informasi layanan.")
            gr.ChatInterface(
                fn=chatbot,
                examples=[
                    "Apa saja hadiah yang tersedia?",
                    "Bagaimana cara memesan buket bunga?",
                    "Berapa estimasi total untuk 2 Birthday Cake?"
                ]
            )

        with gr.Tab("Admin"):
            gr.Markdown("## Dashboard admin")
            admin_pw = gr.Textbox(label="Password admin", type="password")
            admin_login_btn = gr.Button("Masuk sebagai admin")
            admin_msg = gr.Markdown()
            admin_table = gr.Dataframe(headers=["Nomor","Pelanggan","Email","Produk","Jumlah","Penerima","Tanggal","Total","Status","ID internal"], interactive=False)
            admin_refresh = gr.Button("Muat ulang semua pesanan")
            with gr.Row():
                update_id = gr.Textbox(label="Nomor pesanan (contoh JHD-00001)")
                update_status = gr.Dropdown(choices=STATUSES, value=STATUSES[0], label="Status baru")
            update_btn = gr.Button("Perbarui status", variant="primary")
            update_msg = gr.Markdown()
            admin_login_btn.click(admin_login,admin_pw,[admin_state,admin_msg])
            admin_refresh.click(lambda ok: admin_orders() if ok else [],admin_state,admin_table)
            update_btn.click(update_order_status,[update_id,update_status,admin_state],update_msg).then(
                lambda ok: admin_orders() if ok else [],admin_state,admin_table)

    gr.Markdown("---\n**Catatan demo:** database SQLite berada di penyimpanan runtime Colab dan dapat hilang saat runtime direset.")

if __name__ == "__main__":
    demo.launch(
        share=True,
        debug=False,
        theme=gr.themes.Soft(primary_hue="rose", secondary_hue="stone", neutral_hue="stone"),
        css="""
        @import url('https://fonts.googleapis.com/css2?family=Playfair+Display:ital,wght@0,500;0,600;1,500&family=DM+Sans:wght@400;500;600;700&display=swap');
        * { box-sizing: border-box; }
        body, .gradio-container { background: #efeee8 !important; font-family: 'DM Sans', sans-serif !important; color: #3d493f !important; }
        .gradio-container { max-width: 1250px !important; padding: 22px !important; }
        .website { max-width: 1160px; margin: auto; background: #faf9f5; box-shadow: 0 20px 60px #52645720; }
        .navbar { min-height: 85px; padding: 15px 5%; display: flex; align-items: center; justify-content: space-between; gap: 20px; }
        .brand { display: flex; align-items: center; gap: 12px; font-family: 'Playfair Display', serif; font-size: 15px; letter-spacing: 2px; }
        .brand-icon { width: 43px; height: 43px; border: 1px solid #87998a; border-radius: 50%; display: grid; place-items: center; font-size: 23px; }
        .brand small { display: block; font: 8px 'DM Sans', sans-serif; letter-spacing: 3px; }
        .navbar nav { display: flex; gap: 30px; }
        .navbar nav a { color: #59665e; font-size: 10px; font-weight: 600; letter-spacing: 1.5px; text-decoration: none; }
        .nav-love { font-size: 9px; letter-spacing: 2px; color: #b47d89; }
        .hero { display: grid; grid-template-columns: 0.88fr 1.12fr; min-height: 530px; }
        .hero-content { padding: 75px 10%; background: #fbfaf6; display: flex; flex-direction: column; align-items: flex-start; justify-content: center; }
        .eyebrow { font-size: 9px; font-weight: 700; letter-spacing: 3px; color: #829587; margin-bottom: 25px; }
        .hero h1 { font-family: 'Playfair Display', serif; font-size: clamp(42px, 5vw, 66px); line-height: 1.12; font-weight: 500; letter-spacing: -2px; color: #35453a; margin: 0 0 22px; }
        .hero h1 i { color: #c18a98; font-weight: 500; }
        .hero-content p { font-size: 13px; line-height: 1.9; color: #82877e; max-width: 330px; margin-bottom: 30px; }
        .hero-button { padding: 15px 25px; background: #8da295; color: white; text-decoration: none; border-radius: 30px; font-size: 9px; font-weight: 700; letter-spacing: 1.5px; transition: 0.3s; }
        .hero-button:hover { background: #607968; transform: translateY(-3px); }
        .hero-button span { margin-left: 15px; }
        .hero-image { min-height: 530px; position: relative; background: linear-gradient(0deg, #24382e30, transparent 60%), url('https://images.unsplash.com/photo-1490750967868-88aa4486c946?auto=format&fit=crop&w=1400&q=90'); background-position: center; background-size: cover; }
        .image-label { position: absolute; top: 30px; left: 30px; color: white; font-size: 9px; letter-spacing: 2px; }
        .image-caption { position: absolute; right: 35px; bottom: 35px; color: white; font: 40px 'Playfair Display', serif; text-align: right; }
        .image-caption i { color: #f2d2dc; }
        .running-text { background: #a9b9ad; color: white; text-align: center; padding: 18px; font-size: 9px; letter-spacing: 2px; }
        .collection-intro { display: flex; justify-content: space-between; align-items: flex-end; gap: 30px; padding: 65px 8% 40px; }
        .collection-intro h2 { font: 38px/1.3 'Playfair Display', serif; color: #35453a; }
        .collection-intro h2 i { color: #c18a98; }
        .collection-intro p { max-width: 310px; color: #82877e; font-size: 12px; line-height: 1.9; }
        .catalog-heading { text-align: center; padding: 45px 10px 30px; }
        .catalog-heading span { font-size: 9px; letter-spacing: 3px; color: #87998a; }
        .catalog-heading h2 { font: 36px 'Playfair Display', serif; color: #35453a; margin: 15px 0 5px; }
        .catalog-heading p { color: #858980; font-size: 12px; }
        .product-grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 25px; padding: 15px 3% 35px; }
        .product-card { background: white; padding: 12px; transition: 0.3s; border: 1px solid #eeece5; }
        .product-card:hover { transform: translateY(-7px); box-shadow: 0 15px 35px #52645718; }
        .product-image { height: 230px; position: relative; overflow: hidden; background: #eeeee7; }
        .product-image img { width: 100%; height: 100%; object-fit: cover; transition: 0.5s; }
        .product-card:hover img { transform: scale(1.06); }
        .product-tag { position: absolute; top: 12px; left: 12px; background: #faf9f0dd; color: #75877a; font-size: 8px; letter-spacing: 1px; padding: 8px 10px; }
        .product-info { padding: 15px 5px 8px; }
        .product-info h3 { font: 22px 'Playfair Display', serif; color: #3d493f; }
        .product-info p { font-size: 11px; color: #858980; line-height: 1.7; }
        .product-price { color: #b47d89; font-weight: 700; font-size: 14px; margin-top: 15px; }
        .delivery-note { text-align: center; color: #829587; padding: 25px; font-size: 11px; letter-spacing: 1px; }
        .tabs { max-width: 1160px; margin: 25px auto 0 !important; background: #faf9f5; padding: 20px 4% 40px; }
        .tab-nav button { border-radius: 25px !important; font-size: 11px !important; }
        .tab-nav button.selected { background: #8da295 !important; color: white !important; }
        .gr-button-primary { background: #8da295 !important; border: none !important; border-radius: 25px !important; }
        input, textarea { border-radius: 9px !important; }
        footer { display: none !important; }
        @media(max-width: 768px) {
            .gradio-container { padding: 8px !important; }
            .navbar { flex-wrap: wrap; }
            .navbar nav { gap: 12px; }
            .nav-love { display: none; }
            .hero { grid-template-columns: 1fr; }
            .hero-content { padding: 50px 8%; }
            .hero-image { min-height: 350px; }
            .collection-intro { display: block; padding: 40px 8%; }
            .product-grid { grid-template-columns: 1fr; }
        }
        """
    )


Overwriting app.py


In [ ]:
!python app.py

config.json: 100% 838/838 [00:00<00:00, 2.61MB/s]
tokenizer_config.json: 100% 47.0k/47.0k [00:00<00:00, 78.3MB/s]

tokenizer.json: downloading bytes:  15% 2.55M/17.5M [00:00<00:02, 5.11MB/s]
tokenizer.json: downloading bytes: 100% 7.50M/7.50M [00:00<00:00, 11.8MB/s,  739kB/s  ]
tokenizer.json: reconstructing file: 100% 17.5M/17.5M [00:00<00:00, 27.6MB/s, 1.73MB/s  ]
special_tokens_map.json: 100% 636/636 [00:00<00:00, 2.42MB/s]
Gemma belum dimuat otomatis: GPU tidak terdeteksi. Di Google Colab pilih Runtime > Change runtime type > T4 GPU, lalu jalankan ulang.
/content/app.py:174: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="rose", secondary_hue="emerald"),
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://8b04ec28cf53d8d4b4.gradio.live

This share link is temporary and will last for